## 04 - Transcript Parsing

This notebook reads our earnings call transcripts and extracts
just the CEO and CFO speaking sections.

### Why do we extract only CEO and CFO sections?
- We want to measure what company leadership chose to say
- Analyst Q&A introduces outside voices we don't control for
- CEO language is the most market-moving part of any earnings call

### How it works?
1. Read each transcript text file
2. Find where the CEO starts speaking
3. Extract everything until the Q&A begins
4. Save the cleaned sections for NLP analysis

In [1]:
# Import libraries
import os
import re

# Read one transcript first to understand its structure
with open("../data/transcripts/AAPL_Q1_2020.txt", "r", encoding="utf-8") as f:
    transcript = f.read()

# Print the first 2000 characters to see the structure
print(transcript[:2000])

Skip to main content

Enable accessibility for low vision

Open the accessibility menu
Search for a company
Search for a company
Search for a company
▲ S&P 500 +207% | ▲ Stock Advisor +998% Join The Motley Fool
Accessibility
Log In
Help
The Motley Fool

Our Services
All ServicesStock AdvisorEpicEpic PlusFool PortfoliosFool OneAll PodcastsMotley Fool Money PodcastRule Breaker Investing PodcastThe Motley Fool Foundation

Stock Market News
Live CoverageTrending NewsStock Market NewsMarket MoversTech Stock NewsMarket TrendsCrypto NewsStock Market Indexes TodayMost Active Stocks TodayToday's Biggest Stock GainersToday's Biggest Stock LosersStock Quotes by ExchangeMarket Research📨 Breakfast NewsTop Stocks to Buy NowBest ETFs to BuyBest AI StocksBest Growth StocksDividend KingsBest Index FundsNext Cryptos to ExplodeTechnologyEnergyReal EstateHealthcareConsumer GoodsMaterialsIndustrials

How to Invest
How to Invest MoneyWhat to Invest InHow to Invest in StocksHow to Invest in ETFsHow to Invest

In [2]:
# Find where the actual transcript content begins
# Search for "Tim Cook" or "Operator" which is how transcripts start

start_markers = ["Tim Cook", "Operator", "OPERATOR"]

for marker in start_markers:
    position = transcript.find(marker)
    if position != -1:
        print(f"Found '{marker}' at position {position}")
        print("\nText around that position:")
        print(transcript[position:position+500])
        print("---")
        break

Found 'Tim Cook' at position 4133

Text around that position:
Tim Cook, and he will be followed by CFO, Luca Maestri. After that, we'll open the call to questions from analysts.

Please note that some of the information you'll hear during our discussion today will consist of forward-looking statements, including without limitation those regarding revenue, gross margin, operating expenses, other income and expenses, taxes, capital allocation and future business outlook. Actual results or trends could differ materially from our forecast. For more information
---


In [3]:
# Extract transcript from where Tim Cook is first mentioned
start_position = transcript.find("Tim Cook")
clean_transcript = transcript[start_position:]

# Now find where Q&A begins and stop there
qa_markers = ["Question-and-Answer", "Q&A", "QUESTION AND ANSWER", "Questions and Answers"]

end_position = len(clean_transcript)  # default to end

for marker in qa_markers:
    pos = clean_transcript.find(marker)
    if pos != -1:
        end_position = pos
        print(f"Found Q&A section at position {pos}")
        break

# Extract just the prepared remarks section
prepared_remarks = clean_transcript[:end_position]

print(f"Total characters in clean transcript: {len(prepared_remarks)}")
print("\nFirst 1000 characters:")
print(prepared_remarks[:1000])

Found Q&A section at position 19372
Total characters in clean transcript: 19372

First 1000 characters:
Tim Cook, and he will be followed by CFO, Luca Maestri. After that, we'll open the call to questions from analysts.

Please note that some of the information you'll hear during our discussion today will consist of forward-looking statements, including without limitation those regarding revenue, gross margin, operating expenses, other income and expenses, taxes, capital allocation and future business outlook. Actual results or trends could differ materially from our forecast. For more information, please refer to the risk factors discussed in Apple's most recently filed periodic reports on Form 10-K and Form 10-Q and the Form 8-K filed with the SEC today, along with the associated press release. Apple assumes no obligation to update any forward-looking statements or information, which speaks as of their respective dates.

I'd now like to turn the call over to Tim for introductory rema

In [4]:
# Extract just Tim Cook's section
# CEO section starts after "Tim Cook -- Chief Executive Officer"
# and ends when Luca Maestri (CFO) starts speaking

ceo_start = prepared_remarks.find("Tim Cook -- Chief Executive Officer")
cfo_start = prepared_remarks.find("Luca Maestri")

if ceo_start != -1 and cfo_start != -1:
    ceo_section = prepared_remarks[ceo_start:cfo_start]
    print(f"CEO section length: {len(ceo_section)} characters")
    print("\nCEO section preview:")
    print(ceo_section[:1000])
else:
    print("Could not find CEO or CFO markers")

CEO section length: 0 characters

CEO section preview:



In [5]:
# Let's search for what's actually in the transcript
# Print lines 50 to 80 to find the exact speaker labels

lines = prepared_remarks.split("\n")
for i, line in enumerate(lines[50:100]):
    print(f"Line {i+50}: {line}")

Line 50: Turning to services. We set an all-time revenue record of $12.7 billion with double-digit growth in all of our five geographic segments. As Tim mentioned, we established new all-time records for Apple Music, cloud services, payment services and our App Store search ad business and December quarter records for the App Store and Apple Care. We are well on our way to accomplishing our goal of doubling our fiscal year '16 Services revenue during 2020. We've actually already reached that goal on a run rate basis with the results of the December quarter.
Line 51: 
Line 52: Customer engagement in our ecosystem continues to grow and the number of both transacting and paid accounts on our digital content stores reached a new all-time high with paid accounts growing double-digits in all of our geographic segments. We now have over 480 million paid subscriptions across the services on our platform, up 120 million from a year ago. And at this point, we expect to hit our goal of surpassing

In [6]:
# Find the exact speaker labels in this transcript
for i, line in enumerate(lines):
    if "Cook" in line or "Maestri" in line or "CEO" in line or "Chief" in line:
        print(f"Line {i}: {line}")

Line 0: Tim Cook, and he will be followed by CFO, Luca Maestri. After that, we'll open the call to questions from analysts.
Line 6: Tim Cook -- Chief Executive Officer
Line 40: Luca Maestri -- Senior Vice President & Chief Financial Officer


In [7]:
# Extract CEO section using exact labels found
ceo_marker = "Tim Cook -- Chief Executive Officer"
cfo_marker = "Luca Maestri -- Senior Vice President & Chief Financial Officer"

ceo_start = prepared_remarks.find(ceo_marker)
cfo_start = prepared_remarks.find(cfo_marker)

ceo_section = prepared_remarks[ceo_start:cfo_start]

print(f"CEO section length: {len(ceo_section)} characters")
print("\nCEO section preview:")
print(ceo_section[:1000])

CEO section length: 9412 characters

CEO section preview:
Tim Cook -- Chief Executive Officer

Thanks, Tejas. Good afternoon, and thanks to all of you for joining us. We're thrilled to report Apple's biggest quarter ever, which set new all-time records in both revenue and earnings. We generated revenue of $91.8 billion, which is above the high end of our guidance range, with revenue growth accelerating for the third consecutive quarter. Geographically, we set all-time records in the Americas, Europe and Rest of Asia Pacific, and solid Greater China returned to growth. Our record performance was fueled by iPhone where December quarter revenue was up 8% year-over-year and by our fifth consecutive quarter of double-digit growth outside of iPhone, including a new all-time record for Services and another blowout quarter for Wearables.

Our active installed base of devices has now surpassed 1.5 billion, up over 100 million in the last 12 months alone reaching a new all-time high for each of 

In [8]:
# Build a reusable function to extract CEO section from any transcript

def extract_ceo_section(filepath):
    """
    Reads a transcript file and extracts the CEO speaking section.
    Returns the clean CEO text.
    """
    # Read the file
    with open(filepath, "r", encoding="utf-8") as f:
        transcript = f.read()
    
    # Find where transcript content starts
    start = transcript.find("Tim Cook") 
    if start == -1:
        start = transcript.find("Operator")
    if start == -1:
        start = 0
    
    transcript = transcript[start:]
    
    # Find Q&A section and cut there
    qa_markers = ["Question-and-Answer", "Q&A Session", 
                  "QUESTION AND ANSWER", "Questions and Answers"]
    end = len(transcript)
    for marker in qa_markers:
        pos = transcript.find(marker)
        if pos != -1:
            end = pos
            break
    
    transcript = transcript[:end]
    
    # Split into lines and find speaker labels
    lines = transcript.split("\n")
    speakers = []
    for i, line in enumerate(lines):
        if "Chief Executive Officer" in line or "CEO" in line:
            speakers.append(("CEO", i))
        elif "Chief Financial Officer" in line or "CFO" in line:
            speakers.append(("CFO", i))
    
    print(f"Speakers found: {speakers}")
    return transcript

# Test it on our first transcript
result = extract_ceo_section("../data/transcripts/AAPL_Q1_2020.txt")
print(f"\nExtracted {len(result)} characters")

Speakers found: [('CFO', 0), ('CEO', 6), ('CFO', 40)]

Extracted 19372 characters


In [9]:
# Run on all 5 transcripts and save cleaned versions

transcripts = [
    "AAPL_Q1_2020.txt",
    "AAPL_Q2_2020.txt",
    "MSFT_Q4_2022.txt",
    "NVDA_Q2_2023.txt",
    "AMZN_Q1_2025.txt"
]

for filename in transcripts:
    filepath = f"../data/transcripts/{filename}"
    
    try:
        result = extract_ceo_section(filepath)
        
        # Save cleaned version
        clean_filename = filename.replace(".txt", "_clean.txt")
        clean_filepath = f"../data/transcripts/{clean_filename}"
        
        with open(clean_filepath, "w", encoding="utf-8") as f:
            f.write(result)
        
        print(f"✓ {filename} → {clean_filename} ({len(result)} chars)")
    
    except Exception as e:
        print(f"✗ {filename} failed: {e}")

Speakers found: [('CFO', 0), ('CEO', 6), ('CFO', 40)]
✓ AAPL_Q1_2020.txt → AAPL_Q1_2020_clean.txt (19372 chars)
Speakers found: [('CFO', 0), ('CEO', 6), ('CFO', 42)]
✓ AAPL_Q2_2020.txt → AAPL_Q2_2020_clean.txt (24970 chars)
Speakers found: [('CEO', 18), ('CFO', 80), ('CEO', 176), ('CEO', 200), ('CFO', 228), ('CFO', 250), ('CEO', 270), ('CFO', 276), ('CFO', 296), ('CEO', 304), ('CFO', 322), ('CFO', 346), ('CFO', 358), ('CEO', 362), ('CEO', 375), ('CFO', 377)]
✓ MSFT_Q4_2022.txt → MSFT_Q4_2022_clean.txt (59393 chars)
Speakers found: [('CFO', 12), ('CFO', 14), ('CFO', 60), ('CFO', 78), ('CEO', 86), ('CEO', 108), ('CFO', 140), ('CEO', 150), ('CFO', 168), ('CFO', 170), ('CEO', 178), ('CFO', 196), ('CEO', 206), ('CFO', 222), ('CFO', 240), ('CFO', 258), ('CEO', 266), ('CFO', 284), ('CFO', 302), ('CFO', 316), ('CEO', 324), ('CEO', 346), ('CEO', 362), ('CFO', 387), ('CEO', 391)]
✓ NVDA_Q2_2023.txt → NVDA_Q2_2023_clean.txt (53399 chars)
Speakers found: [('CEO', 8)]
✓ AMZN_Q1_2025.txt → AMZN_Q1_2

In [10]:
import os

files = os.listdir("../data/transcripts")
txt_files = [f for f in files if f.endswith(".txt") and "clean" not in f]

print(f"Total transcripts: {len(txt_files)}")
for f in sorted(txt_files):
    print(f"✓ {f}")

Total transcripts: 10
✓ AAPL_Q1_2020.txt
✓ AAPL_Q2_2020.txt
✓ AMZN_Q1_2020.txt
✓ AMZN_Q1_2025.txt
✓ JNJ_Q1_2020.txt
✓ JPM_Q1_2020.txt
✓ MSFT_Q1_2020.txt
✓ MSFT_Q4_2022.txt
✓ NVDA_Q2_2023.txt
✓ XOM_Q1_2020.txt


In [11]:
# Clean the 5 new transcripts
new_transcripts = [
    "AMZN_Q1_2020.txt",
    "JNJ_Q1_2020.txt",
    "JPM_Q1_2020.txt",
    "MSFT_Q1_2020.txt",
    "XOM_Q1_2020.txt"
]

for filename in new_transcripts:
    filepath = f"../data/transcripts/{filename}"
    
    try:
        result = extract_ceo_section(filepath)
        
        clean_filename = filename.replace(".txt", "_clean.txt")
        clean_filepath = f"../data/transcripts/{clean_filename}"
        
        with open(clean_filepath, "w", encoding="utf-8") as f:
            f.write(result)
        
        print(f"✓ {filename} → {clean_filename} ({len(result)} chars)")
    
    except Exception as e:
        print(f"✗ {filename} failed: {e}")

Speakers found: [('CFO', 6), ('CFO', 12)]
✓ AMZN_Q1_2020.txt → AMZN_Q1_2020_clean.txt (8501 chars)
Speakers found: [('CEO', 8), ('CEO', 20), ('CFO', 160)]
✓ JNJ_Q1_2020.txt → JNJ_Q1_2020_clean.txt (64853 chars)
Speakers found: [('CEO', 2), ('CFO', 4)]
✓ JPM_Q1_2020.txt → JPM_Q1_2020_clean.txt (23353 chars)
Speakers found: [('CEO', 10), ('CEO', 20), ('CFO', 76)]
✓ MSFT_Q1_2020.txt → MSFT_Q1_2020_clean.txt (26302 chars)
Speakers found: [('CEO', 10), ('CEO', 62)]
✓ XOM_Q1_2020.txt → XOM_Q1_2020_clean.txt (25888 chars)
